# video-ai — End-to-End Pipeline Test (Colab)

Runs the full pipeline on an uploaded video: scene detect → motion/audio/faces/objects → CLIP zero-shot → OCR → pose → saliency → depth → BLIP captions → Qwen3-VL scene Q&A → narrative paragraph.

**Recommended runtime**: T4 (free) or A100 (paid). Set `Runtime → Change runtime type → GPU`.

## 1. GPU check

In [1]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('cuda ver:', torch.version.cuda)
print('torch:', torch.__version__)

Sun Apr 26 20:13:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the repo

**Option A** — clone from your GitHub (replace URL):

In [ ]:
%cd /content
!rm -rf video-ai
!git clone https://github.com/<your-user>/video-ai.git
%cd /content/video-ai

**Option B** — upload local zip of the repo:
1. Locally: zip the `video-ai/` folder.
2. Run cell, click 'Choose Files', upload zip, then unzip.

In [2]:
# Skip if you used Option A
from google.colab import files
import os, zipfile
%cd /content
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content')
        # find extracted folder containing pyproject.toml
        for root, dirs, files_ in os.walk('/content'):
            if 'pyproject.toml' in files_:
                print('repo:', root)
                %cd $root
                break
        break

/content


KeyboardInterrupt: 

## 3. Install pipeline

One command. ~3 min first time. Subsequent runs skip download.

In [ ]:
!apt-get install -y ffmpeg > /dev/null
!pip install --upgrade pip wheel setuptools > /dev/null
!pip install -e . 2>&1 | tail -5

## 4. Upload a test video

MP4/MOV/WebM. 2–3 min recommended for full Tier 3 in reasonable time.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded))
print('video:', VIDEO_PATH)
!ls -lh $VIDEO_PATH

## 5. Run pipeline (Tier 1 + 2, no Video-LLM)

Fastest path. Skips Qwen3-VL. Still gets BLIP captions, scene cards, narrative paragraph, OCR, depth, saliency, action, tracking.

In [ ]:
!python -m analysis "$VIDEO_PATH" --out /content/features_no_vlm.json 2>&1 | tail -30

## 6. Inspect features (no Video-LLM)

In [ ]:
import json, textwrap
vf = json.load(open('/content/features_no_vlm.json'))
print(f"video_id     : {vf['video_id']}")
print(f"duration     : {vf['duration']:.2f}s")
print(f"segments     : {len(vf['timeline'])}")
print(f"highlights   : {len(vf['highlights'])}")
print(f"global_dec   : {vf.get('global_decisions', [])}")
print('\n=== narrative paragraph ===')
print(textwrap.fill(vf.get('narrative', '(empty)'), 100))
print('\n=== first scene card ===')
print(json.dumps(vf['timeline'][0].get('scene_card', {}), indent=2)[:1500])
print('\n=== highlights ===')
for h in vf['highlights']:
    print(f"  {h['t0']:.2f}-{h['t1']:.2f}s  score={h['score']:.2f}")

## 7. Run with Video-LLM (Qwen3-VL-4B-Instruct)

Heavy. First run downloads ~9 GB. Needs ≥8 GB GPU.
Drops `--no-depth --no-captions` to free VRAM since Qwen3-VL covers same ground better.

In [ ]:
!python -m analysis "$VIDEO_PATH" \
  --out /content/features_vlm.json \
  --video-llm \
  --video-llm-backend qwen2vl \
  --video-llm-model 'Qwen/Qwen3-VL-4B-Instruct' \
  --video-llm-no-4bit \
  --no-depth --no-captions 2>&1 | tail -30

## 8. Inspect Video-LLM output

In [ ]:
import json, textwrap
vf = json.load(open('/content/features_vlm.json'))
print('=== whole-video summary (Qwen3-VL) ===')
print(textwrap.fill(vf.get('vlm_video_summary') or '(empty)', 100))
print('\n=== narrative paragraph (VLM-augmented) ===')
print(textwrap.fill(vf.get('narrative', ''), 100))
print('\n=== per-scene VLM cards ===')
shown = set()
for seg in vf['timeline']:
    f = seg['features']
    if not f.get('vlm_summary'):
        continue
    key = f['vlm_summary']
    if key in shown:
        continue
    shown.add(key)
    print(f"\n[{seg['t0']:.1f}-{seg['t1']:.1f}s]")
    print(f"  summary : {f.get('vlm_summary')}")
    print(f"  action  : {f.get('vlm_action')}")
    print(f"  subjects: {f.get('vlm_subjects')}")
    print(f"  setting : {f.get('vlm_setting')}")
    print(f"  mood    : {f.get('vlm_mood')}")

## 9. Run backend service + test endpoints

In [ ]:
import subprocess, time
p = subprocess.Popen(['uvicorn', 'backend.app.main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(6)
!curl -s http://127.0.0.1:8000/health
print()

In [ ]:
# Reuse cached video_id (same file, same mtime → same id)
VID = vf['video_id']
print('video_id:', VID)
!curl -s "http://127.0.0.1:8000/narrative/$VID?style=summary"
print()
!curl -s "http://127.0.0.1:8000/narrative/$VID?style=bullets" | python -m json.tool

In [ ]:
# Generate edit plan + render reel
import json, requests
r = requests.post('http://127.0.0.1:8000/edit-plan',
                  json={'video_id': VID, 'mode': 'reel'}).json()
print(json.dumps(r, indent=2)[:1500])

In [ ]:
# Render reel mp4
import requests
with requests.post('http://127.0.0.1:8000/render',
                   json={'video_id': VID, 'mode': 'reel'},
                   stream=True) as r:
    with open('/content/reel.mp4', 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
!ls -lh /content/reel.mp4
from IPython.display import Video
Video('/content/reel.mp4', embed=True)

## 10. Cleanup

Stop backend when done:

In [ ]:
p.terminate()